## RealMLP on Predicting Heart Disease

**Score**
- PB:0.95396
- CV:0.955718

**Note**
- Remove noise modeling from the notebook 
- Have 3 models ensembling 
- Change rank stacking from "weight search" to "meta-learning".

### Package import

In [1]:
!pip install pytabkit -q

from pathlib import Path
import json
import zipfile
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import warnings
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# ---- Config ----
COMP_SLUG = "playground-series-s6e2"
KAGGLE_COMP_DIR = Path("/kaggle/input/competitions/playground-series-s6e2")
KAGGLE_EXT_PATH = Path("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv")

LOCAL_DATA_DIR = Path("data/raw")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

NEED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.0 MB/s eta 0:00:00


In [2]:
def run(cmd: list[str]) -> None:
    p = subprocess.run(cmd, capture_output=True, text=True)
    p.check_returncode()


def ensure_kaggle_cli() -> None:
    try:
        pass
    except Exception:
        subprocess.check_call(["pip", "-q", "install", "kaggle"])


def ensure_kaggle_json_interactive_colab(dst: Path = Path("/content/kaggle.json")) -> Path:
    """
    In Colab: open upload dialog if /content/kaggle.json is missing.
    In non-Colab: just require the file to exist.
    """
    if dst.exists():
        print("Found:", dst)
        return dst

    try:
        from google.colab import files  # type: ignore
    except Exception:
        raise FileNotFoundError(
            f"{dst} not found. Please place kaggle.json at {dst} (Colab) "
            "or provide credentials another way."
        )

    print("Upload your kaggle.json (Kaggle -> Account -> API -> Create New Token)")
    uploaded = files.upload()
    cand = None
    if "kaggle.json" in uploaded:
        cand = "kaggle.json"
    else:
        for name in uploaded.keys():
            if name.endswith(".json"):
                cand = name
                break
    if cand is None:
        raise FileNotFoundError("Upload failed: no .json file received.")

    Path(cand).rename(dst)
    print("Saved to:", dst)
    return dst


def install_kaggle_json(src: Path) -> None:
    """
    Copy /content/kaggle.json -> ~/.kaggle/kaggle.json (chmod 600)
    """
    if not src.exists():
        raise FileNotFoundError(f"{src} not found.")

    dst_dir = Path.home() / ".kaggle"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "kaggle.json"

    dst.write_bytes(src.read_bytes())
    try:
        dst.chmod(0o600)
    except Exception:
        pass

    cfg = json.loads(dst.read_text())
    if "username" not in cfg or "key" not in cfg:
        raise ValueError("kaggle.json is missing 'username' or 'key'.")
    print(f"Installed kaggle.json for user: {cfg['username']}")


def local_data_ready(data_dir: Path) -> bool:
    return all((data_dir / f).exists() for f in NEED_FILES)


def download_competition_to(data_dir: Path) -> None:
    """
    Download competition zip(s) and extract into data_dir.
    Assumes kaggle CLI + credentials are ready.
    """
    run(["kaggle", "config", "view"])
    run(["kaggle", "competitions", "download", "-c", COMP_SLUG, "-p", str(data_dir), "--force"])

    zips = list(data_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip found in {data_dir} after download.")

    for zp in zips:
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(data_dir)
        print("Unzipped:", zp.name)

    if not local_data_ready(data_dir):
        missing = [f for f in NEED_FILES if not (data_dir / f).exists()]
        raise FileNotFoundError(f"Download/unzip finished but missing: {missing}")


In [3]:
if KAGGLE_COMP_DIR.exists():
    DATA_SRC = "kaggle"
    data_dir = KAGGLE_COMP_DIR
    print("Using Kaggle mounted competition data:", data_dir)
else:
    DATA_SRC = "local"
    data_dir = LOCAL_DATA_DIR
    if local_data_ready(data_dir):
        print("Using local data (already present):", data_dir)
    else:
        print("Local data missing -> download using kaggle.json")
        ensure_kaggle_cli()
        kaggle_json_src = ensure_kaggle_json_interactive_colab(Path("/content/kaggle.json"))
        install_kaggle_json(kaggle_json_src)
        download_competition_to(data_dir)
        print("Download complete -> using local data:", data_dir)


# ---- Load ----
train = pd.read_csv(data_dir / "train.csv")
test  = pd.read_csv(data_dir / "test.csv")
sub   = pd.read_csv(data_dir / "sample_submission.csv")

# external dataset: only available if mounted on Kaggle; optional
original = pd.read_csv(KAGGLE_EXT_PATH) if KAGGLE_EXT_PATH.exists() else None

print("train:", train.shape, "test:", test.shape, "sub:", sub.shape, "original:", None if original is None else original.shape)
print("DATA_SRC:", DATA_SRC)

# External data loading
train_comp = train.copy()

# Encode target with LabelEncoder if not numeric
if not pd.api.types.is_numeric_dtype(train_comp["Heart Disease"]):
    le = LabelEncoder()
    train_comp["Heart Disease"] = le.fit_transform(train_comp["Heart Disease"])
    if original is not None and "Heart Disease" in original.columns:
        original["Heart Disease"] = le.transform(original["Heart Disease"])

# Align external columns to train schema
if original is not None:
    if "Heart Disease" not in original.columns:
        raise ValueError("External dataset missing target column: Heart Disease")

    original_aligned = original.copy()
    # add missing columns
    for col in train_comp.columns:
        if col not in original_aligned.columns:
            original_aligned[col] = np.nan

    # ensure id exists
    if "id" not in original_aligned.columns:
        original_aligned["id"] = -(np.arange(len(original_aligned)) + 1)

    # align column order
    original_aligned = original_aligned[train_comp.columns]

    train_full = pd.concat([train_comp, original_aligned], ignore_index=True)
    train_full["is_external"] = [0] * len(train_comp) + [1] * len(original_aligned)
else:
    train_full = train_comp.copy()
    train_full["is_external"] = 0

# use concatenated data for downstream
train = train_full


Using Kaggle mounted competition data: /kaggle/input/competitions/playground-series-s6e2
train: (630000, 15) test: (270000, 14) sub: (270000, 2) original: None
DATA_SRC: kaggle


In [4]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.set_float32_matmul_precision("high")
N_FOLDS = 5
USE_ALL_CAT = True

print(f"Using device: {DEVICE}")


Using device: cuda


### Data download

In [5]:
display(train.head())
display(test.head())
display(sub.head())

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease,is_external
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1,0
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0,0
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0,0
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0,0
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1,0


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [6]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


train: (630000, 16)
test: (270000, 14)
Only in train: ['Heart Disease', 'is_external']
Only in test: []


,dtype
id,int64
Age,int64
Sex,int64
Chest pain type,int64
BP,int64
Cholesterol,int64
FBS over 120,int64
EKG results,int64
Max HR,int64
Exercise angina,int64


### Data Preprocessing

In [7]:
def encode_target_strict(y: pd.Series) -> pd.Series:
    """Map common string labels to {0,1}. Raises if unknown."""
    mapping_candidates = [
        {"No": 0, "Yes": 1},
        {"N": 0, "Y": 1},
        {"Negative": 0, "Positive": 1},
        {"Absent": 0, "Present": 1},
        {"Absence": 0, "Presence": 1},
        {0: 0, 1: 1},
        {"0": 0, "1": 1},
    ]
    uniq = set(pd.Series(y).dropna().unique().tolist())
    for mp in mapping_candidates:
        if uniq.issubset(set(mp.keys())):
            return pd.Series(y).map(mp).astype("int8")
    raise ValueError(f"Unknown target labels: {sorted(list(uniq))}")


# ---- target ----
if not pd.api.types.is_numeric_dtype(train["Heart Disease"]):
    train["Heart Disease"] = encode_target_strict(train["Heart Disease"])
if original is not None and "Heart Disease" in original.columns:
    if not pd.api.types.is_numeric_dtype(original["Heart Disease"]):
        original["Heart Disease"] = encode_target_strict(original["Heart Disease"])

TARGET_COL = "Heart Disease"
ID_COL = "id"
META_COLS = [TARGET_COL, ID_COL, "is_external"]

BASE_FEATURES = [c for c in train.columns if c not in META_COLS]

# Canonical S6E2 semantic categoricals (keep as category for embeddings/encoding)
CANONICAL_CAT = {
    "Sex",
    "Chest pain type",
    "FBS over 120",
    "EKG results",
    "Exercise angina",
    "Slope of ST",
    "Number of vessels fluro",
    "Thallium",
}

def split_cols(df: pd.DataFrame):
    base = [c for c in df.columns if c not in META_COLS]
    cat = [c for c in base if c in CANONICAL_CAT]
    num = [c for c in base if c not in cat]
    return cat, num


def add_external_target_stats(df: pd.DataFrame, original_df: pd.DataFrame | None) -> pd.DataFrame:
    """Merge group-wise target stats from the external/original dataset.
    Safe in Playground comps because original_df labels are not the competition labels.
    """
    if original_df is None:
        return df.copy()

    out = df.copy()
    initial_rows = len(out)

    for col in BASE_FEATURES:
        if col not in original_df.columns:
            continue

        stats = (
            original_df.groupby(col)[TARGET_COL]
            .agg(["mean", "median", "std", "skew", "count"])
            .reset_index()
        )
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ["mean", "median", "std", "skew", "count"]]

        out = out.merge(stats, on=col, how="left")
        if len(out) != initial_rows:
            raise ValueError(f"Merge expanded rows for column {col}! {initial_rows} -> {len(out)}")

        # fill NAs for unseen values
        global_mean = float(original_df[TARGET_COL].mean())
        global_median = float(original_df[TARGET_COL].median())
        fill = {
            f"orig_{col}_mean": global_mean,
            f"orig_{col}_median": global_median,
            f"orig_{col}_std": 0.0,
            f"orig_{col}_skew": 0.0,
            f"orig_{col}_count": 0.0,
        }
        out = out.fillna(value=fill)

    return out



def build_features(
    train_fe: pd.DataFrame,
    test_fe: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
):
    tr = train_fe.copy()
    te = test_fe.copy()

    # Keep only original + orig_* stats (no periodic/frequency encodings)
    print(f"Train Shape after FE: {tr.shape}")
    print(f"Test Shape after FE:  {te.shape}")

    # ---- Build X/y with clean dtypes ----
    drop_tr = [c for c in META_COLS if c in tr.columns]
    drop_te = [c for c in [ID_COL] if c in te.columns]

    X = tr.drop(columns=drop_tr).copy()
    X_test = te.drop(columns=drop_te).copy()
    y = tr[TARGET_COL].copy()

    cat_cols_final = [c for c in X.columns if c in cat_cols]
    num_cols_final = [c for c in X.columns if c not in cat_cols_final]

    # Ensure no overlap
    assert set(cat_cols_final).isdisjoint(set(num_cols_final))

    for c in cat_cols_final:
        combined = pd.concat([X[c], X_test[c]], axis=0, ignore_index=True)
        cats = pd.Categorical(combined).categories
        X[c] = pd.Categorical(X[c], categories=cats)
        X_test[c] = pd.Categorical(X_test[c], categories=cats)

    for c in num_cols_final:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").astype("float32")

    return X, X_test, y, cat_cols_final, num_cols_final


# ---- Feature engineering: match realmlp-ext-target-stats-5-fold-cv ----

# 1) External target stats, same idea as add_engineered_features in the CV notebook
train_fe = add_external_target_stats(train, original)
test_fe  = add_external_target_stats(test, original)

print("Train Shape after FE:", train_fe.shape)
print("Test Shape after FE: ", test_fe.shape)

# 2) Build X, y, X_test just like realmlp-ext-target-stats-5-fold-cv
#    base_features = all columns except target + id
TARGET_COL = "Heart Disease"
ID_COL = "id"

X = train_fe.drop(columns=[ID_COL, TARGET_COL])
y = train_fe[TARGET_COL].astype("int8")
X_test = test_fe.drop(columns=[ID_COL])

# Align columns (train has is_external, test does not)
X = X.drop(columns=["is_external"], errors="ignore")

print("X Shape:", X.shape)
print("X_test Shape:", X_test.shape)

# 3) Optional but close to the CV notebook: treat everything as categorical for RealMLP
#    (In the CV notebook they did astype(str).astype('category') on all columns.)
for col in X.columns:
    X[col] = X[col].astype(str).astype("category")
    X_test[col] = X_test[col].astype(str).astype("category")

# We'll let RealMLP infer categorical columns from dtype, so we don't need cat_cols here.
cat_cols = list(X.columns)


Train Shape after FE: (630000, 16)
Test Shape after FE:  (270000, 14)
X Shape: (630000, 13)
X_test Shape: (270000, 13)


### Data Quality Check

In [8]:
def check_data_quality(df, name="Dataset"):
    print(f"--- Data Quality: {name} ---")
    print(f"Total Rows: {len(df)}")

    cols_to_check = [c for c in df.columns if c != 'id']
    dupes = df.duplicated(subset=cols_to_check).sum()

    nan_counts = df.isnull().sum()
    total_nans = nan_counts.sum()

    print(f"Duplicate Rows (excl. ID): {dupes}")
    print(f"Total NaN values: {total_nans}")
    if total_nans > 0:
        print("\nColumns with NaNs:")
        print(nan_counts[nan_counts > 0])
    print("-" * 30)

check_data_quality(train, "Train")
check_data_quality(test, "Test")

--- Data Quality: Train ---
Total Rows: 630000
Duplicate Rows (excl. ID): 0
Total NaN values: 0
------------------------------
--- Data Quality: Test ---
Total Rows: 270000
Duplicate Rows (excl. ID): 0
Total NaN values: 0
------------------------------


### Feature Uniqueness & Cardinality

In [9]:
# Target distribution analysis

def analyze_uniqueness(df):
    unique_stats = []
    for col in df.columns:
        if col == 'id':
            continue

        n_unique = df[col].nunique()
        dtype = df[col].dtype

        category_guess = "Categorical/Ordinal" if n_unique < 25 else "Continuous"

        if pd.api.types.is_numeric_dtype(df[col]):
            mean_val = float(pd.to_numeric(df[col], errors='coerce').mean())
            std_val = float(pd.to_numeric(df[col], errors='coerce').std())
        else:
            mean_val = float('nan')
            std_val = float('nan')

        unique_stats.append({
            'Feature': col,
            'Unique Values': n_unique,
            'Data Type': dtype,
            'Heuristic Type': category_guess,
            'Mean': mean_val,
            'Std': std_val,
        })

    return pd.DataFrame(unique_stats).sort_values(by='Unique Values')

# class imbalance
comp_mask = (train["is_external"] == 0)
ext_mask  = (train["is_external"] == 1)

print("=== Target distribution: COMP ONLY ===")
print(train.loc[comp_mask, TARGET_COL].value_counts(dropna=False))
print(train.loc[comp_mask, TARGET_COL].value_counts(normalize=True, dropna=False))

print("\n=== Target distribution: EXTERNAL ONLY ===")
print(train.loc[ext_mask, TARGET_COL].value_counts(dropna=False))
print(train.loc[ext_mask, TARGET_COL].value_counts(normalize=True, dropna=False))

print("\n=== Target distribution: MERGED (comp + external) ===")
print(train[TARGET_COL].value_counts(dropna=False))
print(train[TARGET_COL].value_counts(normalize=True, dropna=False))

uniqueness_df = analyze_uniqueness(train)
uniqueness_df


=== Target distribution: COMP ONLY ===
Heart Disease
0    347546
1    282454
Name: count, dtype: int64
Heart Disease
0    0.55166
1    0.44834
Name: proportion, dtype: float64

=== Target distribution: EXTERNAL ONLY ===
Series([], Name: count, dtype: int64)
Series([], Name: proportion, dtype: float64)

=== Target distribution: MERGED (comp + external) ===
Heart Disease
0    347546
1    282454
Name: count, dtype: int64
Heart Disease
0    0.55166
1    0.44834
Name: proportion, dtype: float64


,Feature,Unique Values,Data Type,Heuristic Type,Mean,Std
14,is_external,1,int64,Categorical/Ordinal,0.000000,0.000000
1,Sex,2,int64,Categorical/Ordinal,0.714735,0.451541
8,Exercise angina,2,int64,Categorical/Ordinal,0.273725,0.445870
5,FBS over 120,2,int64,Categorical/Ordinal,0.079987,0.271274
13,Heart Disease,2,int64,Categorical/Ordinal,0.448340,0.497324
10,Slope of ST,3,int64,Categorical/Ordinal,1.455871,0.545192
6,EKG results,3,int64,Categorical/Ordinal,0.981660,0.998783
12,Thallium,3,int64,Categorical/Ordinal,4.618873,1.950007
2,Chest pain type,4,int64,Categorical/Ordinal,3.312752,0.851615
11,Number of vessels fluro,4,int64,Categorical/Ordinal,0.451040,0.798549


### Cross-validation setup


In [10]:
ID_COL = "id"
TARGET_COL = "Heart Disease"

# Competition rows only (is_external == 0)
comp_mask = (train["is_external"] == 0).values

X_comp = X.loc[comp_mask].reset_index(drop=True)
y_comp = y.loc[comp_mask].reset_index(drop=True)
comp_ids = train.loc[comp_mask, ID_COL].reset_index(drop=True)

X_test_comp = X_test.copy()
test_ids = test[ID_COL].reset_index(drop=True)

print("X_comp:", X_comp.shape)
print("y_comp:", y_comp.shape)
print("X_test_comp:", X_test_comp.shape)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


X_comp: (630000, 13)
y_comp: (630000,)
X_test_comp: (270000, 13)


### Base model and training


In [11]:
!pip install catboost -q


In [12]:
from catboost import CatBoostClassifier, Pool

cat_features_idx = [X_comp.columns.get_loc(c) for c in cat_cols]

oof_cat = np.zeros(len(X_comp), dtype="float32")
test_cat = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool = Pool(X_test_comp, cat_features=cat_features_idx)

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE + fold,
        iterations=4000,
        od_type="Iter",
        od_wait=200,
        verbose=200,
        task_type="GPU" if DEVICE == "cuda" else "CPU",
    )

    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    pred_val = model.predict_proba(valid_pool)[:, 1]
    oof_cat[val_idx] = pred_val.astype("float32")

    pred_test = model.predict_proba(test_pool)[:, 1]
    test_cat += (pred_test / N_FOLDS).astype("float32")

cat_oof_auc = roc_auc_score(y_comp, oof_cat)
print(f"[CatBoost] OOF AUC = {cat_oof_auc:.6f}")


[CatBoost] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9406781	best: 0.9406781 (0)	total: 162ms	remaining: 10m 46s
200:	test: 0.9557329	best: 0.9557329 (200)	total: 9.41s	remaining: 2m 57s
400:	test: 0.9559671	best: 0.9559671 (399)	total: 18.7s	remaining: 2m 48s
600:	test: 0.9560020	best: 0.9560020 (599)	total: 28s	remaining: 2m 38s
800:	test: 0.9560084	best: 0.9560101 (723)	total: 37.2s	remaining: 2m 28s
1000:	test: 0.9560127	best: 0.9560128 (991)	total: 46.5s	remaining: 2m 19s
1200:	test: 0.9560164	best: 0.9560167 (1198)	total: 55.8s	remaining: 2m 10s
1400:	test: 0.9560186	best: 0.9560197 (1320)	total: 1m 5s	remaining: 2m 1s
1600:	test: 0.9560170	best: 0.9560230 (1465)	total: 1m 14s	remaining: 1m 52s
bestTest = 0.9560230076
bestIteration = 1465
Shrink model to first 1466 iterations.
[CatBoost] Fold 2/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9390359	best: 0.9390359 (0)	total: 53.2ms	remaining: 3m 32s
200:	test: 0.9545804	best: 0.9545804 (200)	total: 9.42s	remaining: 2m 58s
400:	test: 0.9548077	best: 0.9548077 (400)	total: 18.9s	remaining: 2m 49s
600:	test: 0.9548555	best: 0.9548555 (600)	total: 28.4s	remaining: 2m 40s
800:	test: 0.9548785	best: 0.9548785 (799)	total: 37.8s	remaining: 2m 30s
1000:	test: 0.9548834	best: 0.9548839 (902)	total: 47.1s	remaining: 2m 21s
1200:	test: 0.9548866	best: 0.9548869 (1199)	total: 56.6s	remaining: 2m 11s
1400:	test: 0.9548889	best: 0.9548925 (1354)	total: 1m 5s	remaining: 2m 2s
bestTest = 0.9548924565
bestIteration = 1354
Shrink model to first 1355 iterations.
[CatBoost] Fold 3/5


...

0:	test: 0.9394967	best: 0.9394967 (0)	total: 59.4ms	remaining: 3m 57s
200:	test: 0.9553767	best: 0.9553767 (200)	total: 9.34s	remaining: 2m 56s
400:	test: 0.9556375	best: 0.9556376 (399)	total: 18.8s	remaining: 2m 49s
600:	test: 0.9556866	best: 0.9556866 (600)	total: 28.2s	remaining: 2m 39s
800:	test: 0.9557021	best: 0.9557030 (794)	total: 37.7s	remaining: 2m 30s
bestTest = 0.9557030201
bestIteration = 794
Shrink model to first 795 iterations.
[CatBoost] Fold 4/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9393728	best: 0.9393728 (0)	total: 61ms	remaining: 4m 3s
200:	test: 0.9549219	best: 0.9549219 (200)	total: 9.28s	remaining: 2m 55s
400:	test: 0.9552011	best: 0.9552012 (397)	total: 18.8s	remaining: 2m 48s
600:	test: 0.9552537	best: 0.9552541 (599)	total: 28.2s	remaining: 2m 39s
800:	test: 0.9552748	best: 0.9552754 (797)	total: 37.6s	remaining: 2m 30s
1000:	test: 0.9552893	best: 0.9552893 (1000)	total: 47s	remaining: 2m 20s
1200:	test: 0.9552909	best: 0.9552933 (1078)	total: 56.3s	remaining: 2m 11s
bestTest = 0.9552932978
bestIteration = 1078
Shrink model to first 1079 iterations.
[CatBoost] Fold 5/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9400386	best: 0.9400386 (0)	total: 52.7ms	remaining: 3m 30s
200:	test: 0.9558374	best: 0.9558374 (200)	total: 9.36s	remaining: 2m 56s
400:	test: 0.9560810	best: 0.9560810 (400)	total: 18.7s	remaining: 2m 48s
600:	test: 0.9561263	best: 0.9561263 (600)	total: 28s	remaining: 2m 38s
800:	test: 0.9561453	best: 0.9561458 (787)	total: 37.5s	remaining: 2m 29s
1000:	test: 0.9561590	best: 0.9561592 (997)	total: 47.1s	remaining: 2m 21s
1200:	test: 0.9561587	best: 0.9561622 (1058)	total: 56.6s	remaining: 2m 11s
bestTest = 0.9561621845
bestIteration = 1058
Shrink model to first 1059 iterations.
[CatBoost] OOF AUC = 0.955610


In [13]:
param_grid = {
        # 'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'n_epochs': 100,
        'batch_size': 2**12,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'val_metric_name': '1-auc_ovr',
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.04,
        'ls_eps': 0.011498317194338772,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 4,
        'p_drop': 0.07301419697186451,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }
param_grid['device'] = DEVICE


def make_model(param_grid, cat_cols):
    try:
        return RealMLP_TD_Classifier(**param_grid, cat_col_names=cat_cols)
    except TypeError:
        return RealMLP_TD_Classifier(**param_grid)

In [14]:
test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
oof_realmlp = np.zeros(len(X_comp), dtype="float32")
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")

    X_tr, X_val = X_comp.iloc[train_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[train_idx], y_comp.iloc[val_idx]

    
    model = make_model(param_grid, cat_cols)
    model.fit(X_tr, y_tr.values, X_val, y_val.values)

    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test_comp)[:, 1]

    # OOF を fold ごとに埋める
    oof_realmlp[val_idx] = val_probs.astype("float32")

    # test は fold 平均
    test_realmlp += (test_probs / N_FOLDS).astype("float32")

    score = roc_auc_score(y_val, val_probs)
    fold_scores.append(score)
    print(f"Fold {fold} ROC-AUC: {score:.6f}")

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f"\n[RealMLP] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")
print("Fold scores:", fold_scores)


--- RealMLP Fold 1/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047396
Epoch 2/100: val 1-auc_ovr = 0.044425
Epoch 3/100: val 1-auc_ovr = 0.043934
Epoch 4/100: val 1-auc_ovr = 0.043966
Epoch 5/100: val 1-auc_ovr = 0.043908
Epoch 6/100: val 1-auc_ovr = 0.043883
Epoch 7/100: val 1-auc_ovr = 0.043881
Epoch 8/100: val 1-auc_ovr = 0.043909
Epoch 9/100: val 1-auc_ovr = 0.043943
Epoch 10/100: val 1-auc_ovr = 0.043965
Epoch 11/100: val 1-auc_ovr = 0.043936
Epoch 12/100: val 1-auc_ovr = 0.043983
Epoch 13/100: val 1-auc_ovr = 0.043933
Epoch 14/100: val 1-auc_ovr = 0.043955
Epoch 15/100: val 1-auc_ovr = 0.043933
Epoch 16/100: val 1-auc_ovr = 0.043922
Epoch 17/100: val 1-auc_ovr = 0.043947
Epoch 18/100: val 1-auc_ovr = 0.043961
Epoch 19/100: val 1-auc_ovr = 0.043973
Epoch 20/100: val 1-auc_ovr = 0.043979
Epoch 21/100: val 1-auc_ovr = 0.043983
Epoch 22/100: val 1-auc_ovr = 0.043984
Epoch 23/100: val 1-auc_ovr = 0.044007
Epoch 24/100: val 1-auc_ovr = 0.044003
Epoch 25/100: val 1-auc_ovr = 0.043981
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 ROC-AUC: 0.956119

--- RealMLP Fold 2/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


...

Fold 4 ROC-AUC: 0.955436

--- RealMLP Fold 5/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047628
Epoch 2/100: val 1-auc_ovr = 0.044322
Epoch 3/100: val 1-auc_ovr = 0.043898
Epoch 4/100: val 1-auc_ovr = 0.043818
Epoch 5/100: val 1-auc_ovr = 0.043801
Epoch 6/100: val 1-auc_ovr = 0.043785
Epoch 7/100: val 1-auc_ovr = 0.043774
Epoch 8/100: val 1-auc_ovr = 0.043792
Epoch 9/100: val 1-auc_ovr = 0.043812
Epoch 10/100: val 1-auc_ovr = 0.043804
Epoch 11/100: val 1-auc_ovr = 0.043865
Epoch 12/100: val 1-auc_ovr = 0.043801
Epoch 13/100: val 1-auc_ovr = 0.043811
Epoch 14/100: val 1-auc_ovr = 0.043784
Epoch 15/100: val 1-auc_ovr = 0.043829
Epoch 16/100: val 1-auc_ovr = 0.043795
Epoch 17/100: val 1-auc_ovr = 0.043799
Epoch 18/100: val 1-auc_ovr = 0.043831
Epoch 19/100: val 1-auc_ovr = 0.043840
Epoch 20/100: val 1-auc_ovr = 0.043843
Epoch 21/100: val 1-auc_ovr = 0.043851
Epoch 22/100: val 1-auc_ovr = 0.043857
Epoch 23/100: val 1-auc_ovr = 0.043858
Epoch 24/100: val 1-auc_ovr = 0.043841
Epoch 25/100: val 1-auc_ovr = 0.043835
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 ROC-AUC: 0.956226

[RealMLP] OOF AUC = 0.955657
Fold scores: [np.float64(0.9561193722320546), np.float64(0.9549367578660592), np.float64(0.9557678573264159), np.float64(0.9554355458965172), np.float64(0.9562259481749176)]


### Rename / Noise Feature Fallback


In [15]:
# 既存変数名の整理（必要なら）
oof_real = oof_realmlp.copy()
test_real = test_realmlp.copy()


In [16]:
import numpy as np
import pandas as pd

def safe_array(x, n, fill_value=0.0, name="arr"):
    """x が None / 未定義でも長さ n の配列を返す"""
    if x is None:
        return np.full(n, fill_value, dtype="float64")
    arr = np.asarray(x, dtype="float64")
    if len(arr) != n:
        raise ValueError(f"{name} length mismatch: expected {n}, got {len(arr)}")
    return arr

# 例: oof_p_noise, test_p_noise がある notebook ならそのまま使う
# ない notebook なら 0 埋め
try:
    _oof_p_noise = oof_p_noise
except NameError:
    _oof_p_noise = None

try:
    _test_p_noise = test_p_noise
except NameError:
    _test_p_noise = None

oof_p_noise_feat = safe_array(_oof_p_noise, len(y_comp), fill_value=0.0, name="oof_p_noise")
test_p_noise_feat = safe_array(_test_p_noise, len(X_test_comp), fill_value=0.0, name="test_p_noise")

print("oof_p_noise_feat:", pd.Series(oof_p_noise_feat).describe())
print("test_p_noise_feat:", pd.Series(test_p_noise_feat).describe())


oof_p_noise_feat: count    630000.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64
test_p_noise_feat: count    270000.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64


### XGBoost (Diverse 3rd Model)


In [17]:
!pip install xgboost -q
import xgboost as xgb
from sklearn.metrics import roc_auc_score

def to_xgb_matrix(X_df: pd.DataFrame, X_test_df: pd.DataFrame, cat_cols: list[str]):
    """
    category列を codes 化して XGBoost 用の数値行列を作る。
    train/test でカテゴリ対応を揃える。
    """
    X_tr = X_df.copy()
    X_te = X_test_df.copy()

    for c in cat_cols:
        # train/test 結合して categories を揃える
        combined = pd.concat([X_tr[c], X_te[c]], axis=0, ignore_index=True)
        cats = pd.Categorical(combined).categories

        X_tr[c] = pd.Categorical(X_tr[c], categories=cats).codes.astype("int32")
        X_te[c] = pd.Categorical(X_te[c], categories=cats).codes.astype("int32")

    # 念のため全列数値化
    for c in X_tr.columns:
        X_tr[c] = pd.to_numeric(X_tr[c], errors="coerce").astype("float32")
        X_te[c] = pd.to_numeric(X_te[c], errors="coerce").astype("float32")

    return X_tr, X_te

X_comp_xgb, X_test_xgb = to_xgb_matrix(X_comp, X_test_comp, cat_cols)
print(X_comp_xgb.shape, X_test_xgb.shape)


(630000, 13) (270000, 13)


In [18]:
oof_xgb = np.zeros(len(X_comp_xgb), dtype="float32")
test_xgb = np.zeros(len(X_test_xgb), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp_xgb, y_comp), start=1):
    print(f"[XGB] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp_xgb.iloc[tr_idx], X_comp_xgb.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    # Catと少し違う寄りの設定
    model = xgb.XGBClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=8,
        subsample=0.75,
        colsample_bytree=0.75,
        reg_alpha=0.5,
        reg_lambda=2.0,
        gamma=0.5,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        device="gpu" if DEVICE == "cuda" else "cpu",
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=200,
    )

    pred_val = model.predict_proba(X_val)[:, 1]
    pred_test = model.predict_proba(X_test_xgb)[:, 1]

    oof_xgb[val_idx] = pred_val.astype("float32")
    test_xgb += (pred_test / N_FOLDS).astype("float32")

xgb_oof_auc = roc_auc_score(y_comp, oof_xgb)
print(f"[XGB] OOF AUC = {xgb_oof_auc:.6f}")


[XGB] Fold 1/5
[0]	validation_0-auc:0.91974
[200]	validation_0-auc:0.95400
[400]	validation_0-auc:0.95485
[600]	validation_0-auc:0.95530
[800]	validation_0-auc:0.95556
[1000]	validation_0-auc:0.95570
[1200]	validation_0-auc:0.95576
[1400]	validation_0-auc:0.95580
[1600]	validation_0-auc:0.95583
[1800]	validation_0-auc:0.95583
[2000]	validation_0-auc:0.95583
[2200]	validation_0-auc:0.95582
[2400]	validation_0-auc:0.95581
[2600]	validation_0-auc:0.95579
[2800]	validation_0-auc:0.95578
[3000]	validation_0-auc:0.95577
[3200]	validation_0-auc:0.95576
[3400]	validation_0-auc:0.95574
[3600]	validation_0-auc:0.95572
[3800]	validation_0-auc:0.95570
[4000]	validation_0-auc:0.95569
[4200]	validation_0-auc:0.95567
[4400]	validation_0-auc:0.95565
[4600]	validation_0-auc:0.95563
[4800]	validation_0-auc:0.95561
[4999]	validation_0-auc:0.95559
[XGB] Fold 2/5
[0]	validation_0-auc:0.89475
[200]	validation_0-auc:0.95295
[400]	validation_0-auc:0.95395
[600]	validation_0-auc:0.95435
[800]	validation_0-auc:

In [19]:

# SEEDS = [42, 2024, 2025]

# test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
# oof_realmlp = np.zeros(len(X_comp), dtype="float32")

# for seed in SEEDS:
#     print(f"\n==== SEED {seed} ====")
#     param_grid['random_state'] = seed

#     oof_realmlp = np.zeros(len(X_comp), dtype="float32")
#     test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
#     fold_scores = []

#     for fold, (train_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
#         print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")

#         X_tr, X_val = X_comp.iloc[train_idx], X_comp.iloc[val_idx]
#         y_tr, y_val = y_comp.iloc[train_idx], y_comp.iloc[val_idx]

#         model = make_model(param_grid, cat_cols)
#         model.fit(X_tr, y_tr.values, X_val, y_val.values)

#         val_probs = model.predict_proba(X_val)[:, 1]
#         test_probs = model.predict_proba(X_test_comp)[:, 1]

#         oof_realmlp[val_idx] = val_probs.astype("float32")
#         test_realmlp += (test_probs / N_FOLDS).astype("float32")

#         score = roc_auc_score(y_val, val_probs)
#         fold_scores.append(score)
#         print(f"Fold {fold} ROC-AUC: {score:.6f}")

#         if DEVICE == 'cuda':
#             torch.cuda.empty_cache()

#     print(f"\n[RealMLP][seed={seed}] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")
#     print("Fold scores:", fold_scores)

#     oof_realmlp += oof_realmlp / len(SEEDS)
#     test_realmlp += test_realmlp / len(SEEDS)

# print(f"\n[RealMLP][ensemble] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")


### Evaluation

In [20]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

def rank01(x: np.ndarray) -> np.ndarray:
    """0-1 rank (tiesは単純argsort版; 必要ならrankdataに差し替え可)"""
    x = np.asarray(x, dtype="float64")
    order = np.argsort(x)
    ranks = np.empty_like(order, dtype="float64")
    ranks[order] = np.arange(len(x), dtype="float64")
    return (ranks + 1.0) / (len(x) + 2.0)

def minmax01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype="float64")
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.full_like(x, 0.5)
    return (x - mn) / (mx - mn)

# Base OOF AUC
print("[Base] Cat     OOF AUC =", roc_auc_score(y_comp, oof_cat))
print("[Base] RealMLP OOF AUC =", roc_auc_score(y_comp, oof_real))
if "oof_xgb" in globals():
    print("[Base] XGB     OOF AUC =", roc_auc_score(y_comp, oof_xgb))

print("Spearman(cat, real) =", spearmanr(oof_cat, oof_real).correlation)
if "oof_xgb" in globals():
    print("Spearman(cat, xgb)  =", spearmanr(oof_cat, oof_xgb).correlation)
    print("Spearman(real, xgb) =", spearmanr(oof_real, oof_xgb).correlation)


[Base] Cat     OOF AUC = 0.9556095302279508
[Base] RealMLP OOF AUC = 0.9556570042107372
[Base] XGB     OOF AUC = 0.9552204345460825
Spearman(cat, real) = 0.9980863862317189
Spearman(cat, xgb)  = 0.9972363715087081
Spearman(real, xgb) = 0.9971546202235069


### Meta Features


In [21]:
# ---- 使用するベースモデルを定義 ----
base_oof = {
    "cat": np.asarray(oof_cat, dtype="float64"),
    "real": np.asarray(oof_real, dtype="float64"),
}
base_test = {
    "cat": np.asarray(test_cat, dtype="float64"),
    "real": np.asarray(test_real, dtype="float64"),
}

# 3本目を追加（XGB or Cat_ALT）
if "oof_xgb" in globals():
    base_oof["xgb"] = np.asarray(oof_xgb, dtype="float64")
    base_test["xgb"] = np.asarray(test_xgb, dtype="float64")
elif "oof_cat_alt" in globals():
    base_oof["cat_alt"] = np.asarray(oof_cat_alt, dtype="float64")
    base_test["cat_alt"] = np.asarray(test_cat_alt, dtype="float64")

model_names = list(base_oof.keys())
print("Use models for meta-stacking:", model_names)

# ---- rank特徴・prob特徴を作る ----
meta_oof_df = pd.DataFrame(index=np.arange(len(y_comp)))
meta_test_df = pd.DataFrame(index=np.arange(len(X_test_comp)))

for name in model_names:
    p_oof = base_oof[name]
    p_te = base_test[name]

    meta_oof_df[f"prob_{name}"] = p_oof
    meta_test_df[f"prob_{name}"] = p_te

    meta_oof_df[f"rank_{name}"] = rank01(p_oof)
    meta_test_df[f"rank_{name}"] = rank01(p_te)

# ---- pairwise gap特徴（順位差・確率差）----
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a = model_names[i]
        b = model_names[j]

        meta_oof_df[f"rank_gap_{a}_{b}"] = np.abs(meta_oof_df[f"rank_{a}"] - meta_oof_df[f"rank_{b}"])
        meta_test_df[f"rank_gap_{a}_{b}"] = np.abs(meta_test_df[f"rank_{a}"] - meta_test_df[f"rank_{b}"])

        meta_oof_df[f"prob_gap_{a}_{b}"] = np.abs(meta_oof_df[f"prob_{a}"] - meta_oof_df[f"prob_{b}"])
        meta_test_df[f"prob_gap_{a}_{b}"] = np.abs(meta_test_df[f"prob_{a}"] - meta_test_df[f"prob_{b}"])

# ---- 補助特徴としての p_noise ----
meta_oof_df["p_noise"] = oof_p_noise_feat
meta_test_df["p_noise"] = test_p_noise_feat

# ---- 参考: ベース平均（補助列）----
prob_cols_oof = [f"prob_{m}" for m in model_names]
rank_cols_oof = [f"rank_{m}" for m in model_names]

meta_oof_df["prob_mean"] = meta_oof_df[prob_cols_oof].mean(axis=1)
meta_test_df["prob_mean"] = meta_test_df[[f"prob_{m}" for m in model_names]].mean(axis=1)

meta_oof_df["rank_mean"] = meta_oof_df[rank_cols_oof].mean(axis=1)
meta_test_df["rank_mean"] = meta_test_df[[f"rank_{m}" for m in model_names]].mean(axis=1)

print(meta_oof_df.shape, meta_test_df.shape)
display(meta_oof_df.head())


Use models for meta-stacking: ['cat', 'real', 'xgb']
(630000, 15) (270000, 15)


,prob_cat,rank_cat,prob_real,rank_real,prob_xgb,rank_xgb,rank_gap_cat_real,prob_gap_cat_real,rank_gap_cat_xgb,prob_gap_cat_xgb,rank_gap_real_xgb,prob_gap_real_xgb,p_noise,prob_mean,rank_mean
0,0.997245,0.970786,0.993483,0.939211,0.997464,0.971899,0.031575,0.003761,0.001113,0.000219,0.032687,0.003980,0.0,0.996064,0.960632
1,0.010252,0.103320,0.008669,0.076784,0.007580,0.070173,0.026536,0.001583,0.033148,0.002673,0.006611,0.001090,0.0,0.008834,0.083426
2,0.008778,0.088227,0.009887,0.089938,0.011512,0.109493,0.001711,0.001109,0.021267,0.002733,0.019555,0.001625,0.0,0.010059,0.095886
3,0.049142,0.290594,0.048276,0.287317,0.041397,0.269317,0.003278,0.000865,0.021278,0.007745,0.018000,0.006880,0.0,0.046272,0.282409
4,0.996725,0.964624,0.997424,0.980108,0.998857,0.988584,0.015484,0.000700,0.023960,0.002132,0.008476,0.001432,0.0,0.997669,0.977772


### Meta Learning (Ridge)


In [22]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold

X_meta = meta_oof_df.copy()
X_meta_test = meta_test_df.copy()
y_meta = y_comp.values.astype("float64")

meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 999)

meta_oof_pred = np.zeros(len(X_meta), dtype="float64")
meta_test_pred = np.zeros(len(X_meta_test), dtype="float64")

fold_aucs = []
coef_list = []

for fold, (tr_idx, va_idx) in enumerate(meta_skf.split(X_meta, y_meta), start=1):
    X_tr, X_va = X_meta.iloc[tr_idx], X_meta.iloc[va_idx]
    y_tr, y_va = y_meta[tr_idx], y_meta[va_idx]

    meta_model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    meta_model.fit(X_tr, y_tr)

    pred_va = meta_model.predict(X_va)
    pred_te = meta_model.predict(X_meta_test)

    meta_oof_pred[va_idx] = pred_va
    meta_test_pred += pred_te / meta_skf.n_splits

    fold_auc = roc_auc_score(y_va, pred_va)
    fold_aucs.append(fold_auc)
    coef_list.append(pd.Series(meta_model.coef_, index=X_meta.columns, name=f"fold{fold}"))

    print(f"[META-Ridge] Fold {fold} AUC = {fold_auc:.6f}")

meta_auc = roc_auc_score(y_meta, meta_oof_pred)
print("\n[META-Ridge] OOF AUC =", f"{meta_auc:.6f}")
print("Fold AUCs:", [round(x, 6) for x in fold_aucs])

coef_df = pd.concat(coef_list, axis=1)
coef_summary = pd.DataFrame({
    "coef_mean": coef_df.mean(axis=1),
    "coef_abs_mean": coef_df.abs().mean(axis=1),
    "coef_std": coef_df.std(axis=1),
}).sort_values("coef_abs_mean", ascending=False)

display(coef_summary.head(20))


[META-Ridge] Fold 1 AUC = 0.955273
[META-Ridge] Fold 2 AUC = 0.956433
[META-Ridge] Fold 3 AUC = 0.954844
[META-Ridge] Fold 4 AUC = 0.955892
[META-Ridge] Fold 5 AUC = 0.956151

[META-Ridge] OOF AUC = 0.955714
Fold AUCs: [np.float64(0.955273), np.float64(0.956433), np.float64(0.954844), np.float64(0.955892), np.float64(0.956151)]


,coef_mean,coef_abs_mean,coef_std
prob_real,0.507204,0.507204,0.020413
prob_cat,0.314492,0.314492,0.024972
prob_mean,0.249874,0.249874,0.000604
prob_xgb,-0.072076,0.072076,0.008786
prob_gap_cat_real,-0.047060,0.047305,0.028414
prob_gap_cat_xgb,-0.015106,0.030723,0.035337
rank_gap_cat_xgb,-0.026948,0.026948,0.019038
rank_real,-0.016723,0.016723,0.003303
rank_xgb,0.015594,0.015594,0.004315
rank_gap_real_xgb,-0.015431,0.015431,0.008918


### Comparison


In [23]:
# ---- 比較（既存ブレンド vs メタ学習）----
oof_simple = 0.9 * oof_cat + 0.1 * oof_real
auc_simple = roc_auc_score(y_comp, oof_simple)

rank_blend_ref = np.zeros(len(y_comp), dtype="float64")
for m in model_names:
    rank_blend_ref += rank01(base_oof[m]) / len(model_names)
auc_rank_ref = roc_auc_score(y_comp, rank_blend_ref)

auc_meta = roc_auc_score(y_comp, meta_oof_pred)

print(f"[Cat only]         OOF AUC = {roc_auc_score(y_comp, oof_cat):.6f}")
print(f"[RealMLP only]     OOF AUC = {roc_auc_score(y_comp, oof_real):.6f}")
if "oof_xgb" in globals():
    print(f"[XGB only]         OOF AUC = {roc_auc_score(y_comp, oof_xgb):.6f}")
if "oof_cat_alt" in globals():
    print(f"[Cat_ALT only]     OOF AUC = {roc_auc_score(y_comp, oof_cat_alt):.6f}")

print(f"[Simple 0.9/0.1]   OOF AUC = {auc_simple:.6f}")
print(f"[Rank mean]        OOF AUC = {auc_rank_ref:.6f}")
print(f"[Meta Ridge]       OOF AUC = {auc_meta:.6f}")


[Cat only]         OOF AUC = 0.955610
[RealMLP only]     OOF AUC = 0.955657
[XGB only]         OOF AUC = 0.955220
[Simple 0.9/0.1]   OOF AUC = 0.955644
[Rank mean]        OOF AUC = 0.955672
[Meta Ridge]       OOF AUC = 0.955714


### Submission (Meta Ridge)


In [24]:
# ---- Submission（メタ学習版）----
meta_test_submit = minmax01(meta_test_pred)

submission_meta = pd.DataFrame({
    "id": test_ids,
    "Heart Disease": meta_test_submit,
})
submission_meta.to_csv("submission_meta_ridge.csv", index=False)

meta_oof_df_save = pd.DataFrame({
    "id": comp_ids,
    "y_true": y_comp.values,
    "y_pred_meta_ridge": meta_oof_pred,
})
meta_oof_df_save.to_csv("oof_meta_ridge.csv", index=False)

print("Saved:")
print("  submission_meta_ridge.csv")
print("  oof_meta_ridge.csv")
print(submission_meta.shape)


Saved:
  submission_meta_ridge.csv
  oof_meta_ridge.csv
(270000, 2)
